# A Code Walkthrough of the FIFO Trading Engine

This notebook explains how `engine_fifo.py` works and why it is a practical improvement over both the naive engine and the heap-only version. The key idea is price-time priority: best price wins first, and within the same price level, the oldest order is matched first.

### 1. What FIFO Means in an Order Book

A real matching engine usually follows two rules:

- Price priority: better prices should match first.
- Time priority: among orders at the same price, the oldest order should match first.

That second rule is what FIFO gives us: `first in, first out`.

### 2. The Core Optimization

Instead of storing every order in one large list or one raw heap, `engine_fifo.py` organizes the book in two layers:

- `SortedDict` keeps prices sorted.
- `deque` keeps orders at each price in arrival order.

So the engine can efficiently answer two questions at once:

- What is the best available price?
- Which order at that price arrived first?

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in candidates:
    if (candidate / "engine_fifo.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from engine_fifo import Order, OrderBook

### 3. The Main Data Structure

The `OrderBook` stores:

- `bids`: a `SortedDict` of buy price levels
- `asks`: a `SortedDict` of sell price levels
- each price level contains a `deque` of orders

That gives us sorted prices plus FIFO order inside each price bucket.

In [ ]:
book = OrderBook()
print(type(book.bids), type(book.asks))

### 4. Why This Beats a Plain Heap for Fairness

A heap is excellent for finding the best price quickly, but FIFO matching needs a little more structure. At a single price level, we do not just want any order there, we want the oldest one.

With `deque`, the oldest order is always at the front:

- append new resting orders with `.append()`
- consume matched orders with `.popleft()`

That makes time priority explicit and easy to reason about.

In [ ]:
book = OrderBook()

first_buy = Order("buy", "100.00", 5)
second_buy = Order("buy", "100.00", 7)

book.place_order(first_buy)
book.place_order(second_buy)

queue = book.bids[first_buy.price]
print("Front of FIFO queue:", queue[0])
print("Second in queue:", queue[1])

### 5. Matching Logic

For a new buy order, the engine:

1. Looks up the lowest ask using `asks.peekitem(0)`.
2. Checks whether that ask price crosses the incoming buy price.
3. Uses the first order in that ask queue.
4. Executes as much quantity as possible.
5. Removes fully filled front orders with `popleft()`.
6. Deletes the price level if its queue becomes empty.

The sell path does the same thing on the bid side with `bids.peekitem(-1)`.

In [ ]:
book = OrderBook()

book.place_order(Order("buy", "100.00", 5))
book.place_order(Order("buy", "100.00", 7))
book.place_order(Order("buy", "99.50", 4))

print("Book before sell arrives:")
pprint(book.get_order_book_depth())

book.place_order(Order("sell", "100.00", 8))

print("\nBook after matching sell order:")
pprint(book.get_order_book_depth())

print("\nTrades:")
pprint(book.trades)

### 6. What Got Better Than the Naive Engine

Compared with the naive list-based engine:

- no full re-sort of all orders on every insert
- direct access to best ask and best bid price levels
- direct FIFO behavior within each price level
- cleaner market-depth aggregation because quantities are already grouped by price

### 7. What Got Better Than the Heap Engine

Compared with the heap version, the FIFO engine better models exchange behavior:

- price levels are explicit instead of being spread across heap entries
- time priority inside a price level is natural with `deque`
- best bid and best ask are still cheap to read
- book depth is easier to compute because orders are already grouped by price

### 8. Complexity Intuition

| Operation | Naive Lists | Heap | FIFO with `SortedDict + deque` |
| --- | --- | --- | --- |
| Best price lookup | after sorting | heap root | `peekitem()` |
| Same-price time priority | awkward | awkward | natural |
| Add resting order | append + sort later | `heappush` | insert price level + append queue |
| Match oldest order at best price | manual after sorting | needs extra care | queue front with `popleft()` |

In [ ]:
print("Best bid:", book.best_bid)
print("Best ask:", book.best_ask)
print("Last traded price:", book.last_trading_price)

### 9. Summary

The FIFO engine is a strong design because it combines efficiency with market realism. `SortedDict` gives fast access to the best price levels, and `deque` guarantees fair first-in-first-out matching within each level. That makes it a better fit for a real order book than either repeated full sorting or a heap without explicit price queues.